# When does forcing support multi-year persistence?

This notebook **only reads experiment results and plots them**. Computation lives in `scripts/analysis/timed_ablation.py`.

From the repository root, run:
```bash
uv run scripts/analysis/timed_ablation.py --model m1
```

Each event is a centered-index threshold crossing selected before its eventual duration is known. The centered onset at month 0 is confirmed at month 1; interventions start from that shared state. A window labelled 1–3 removes actions applied at states 1, 2 and 3 (which advance the model to states 2, 3 and 4). All other channels follow the policy on the evolving state. Full control resumes after the window.

The outcome is persistence in the same phase for **13 consecutive centered months**, including onset. Positive plotted effects mean removal reduces persistence. These are effects in the controlled onset cohort, not effects on natural ENSO or pre-onset preparation. Each trained agent contributes equally; intervals describe variation across trained agents. Repeated events within a rollout are not treated as independent statistical replicates.


In [2]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import t

ROOT = Path.cwd() if (Path.cwd() / 'scripts').exists() else Path.cwd().parent
NAME = 'm1'
ARCHIVE = ROOT / 'plots' / NAME / 'timing_interventions' / 'timing_interventions.npz'
OUT = ROOT / 'figs'
EXPORT = True
ready = False
if not ARCHIVE.exists():
    print(f'No experiment data yet: {ARCHIVE}. Run timed_ablation.py first.')
else:
    with np.load(ARCHIVE, allow_pickle=False) as archive:
        data = {key: archive[key] for key in archive.files}
    features = list(map(str, data['features']))
    phases = list(map(str, data['phases']))
    lags = data['lags']
    width = int(data['width'])
    values = -100 * data['per_seed_delta']  # control minus intervention, percentage points
    expected = (len(data['seeds']), len(phases), len(features), len(lags))
    assert values.shape == expected and np.isfinite(values).all(), 'Missing/invalid phase effects; collect more onsets.'
    assert str(data['nino34_smooth_alignment']) == 'centered'
    assert len(values) >= 2, 'At least two trained agents required for across-agent intervals.'
    assert phases == ['el_nino', 'la_nina']
    mean = values.mean(axis=0)
    ci = t.ppf(.975, len(values)-1) * values.std(axis=0, ddof=1) / np.sqrt(len(values))
    windows = [f'{k}–{k+width-1}' for k in lags]
    print('Events per agent [El Niño, La Niña]:', data['event_counts'].tolist())
    print('Control persistence probability per agent:', data['control_probability'].round(3))
    print(str(data['lag_convention']))
    if np.all(values == 0):
        print("No persistence changes observed. Zero-width intervals do not establish zero effect; inspect event counts and saved traces.")
    ready = True

plt.rcParams.update({'font.size': 10, 'pdf.fonttype': 42,
                     'axes.spines.top': False, 'axes.spines.right': False})

def export(fig, name):
    if EXPORT:
        OUT.mkdir(exist_ok=True)
        fig.savefig(OUT / f'{name}.pdf', bbox_inches='tight')


No experiment data yet: /Users/ayman/research/enso-rl/plots/m1/timing_interventions/timing_interventions.npz. Run timed_ablation.py first.


## Primary figure: effects with uncertainty

Rows identify forcing channels; columns separate ENSO phases. Points show the mean paired effect across agents and bars show 95% Student-t intervals. These are pointwise intervals, **not multiple-testing-adjusted significance tests**. A near-zero estimate with a wide interval is inconclusive, rather than evidence that the channel is irrelevant.


In [4]:
if ready:
    fig, axes = plt.subplots(len(features), 2, figsize=(9, 2.2*len(features)),
                             sharex=True, sharey=True, squeeze=False, layout='constrained')
    colors = ['#b64d45', '#3877ac']
    limit = max(float(np.max(np.abs(mean) + ci)) * 1.1, 1.)
    for row, feature in enumerate(features):
        for col, title in enumerate(['El Niño', 'La Niña']):
            ax = axes[row, col]
            ax.axhline(0, color='.45', lw=.8)
            ax.set_ylim(-limit, limit)
            ax.errorbar(np.arange(len(lags)), mean[col,row], yerr=ci[col,row],
                        fmt='o', color=colors[col], capsize=3)
            ax.set_xticks(np.arange(len(lags)), windows)
            ax.set_title(f'{chr(97+2*row+col)}  {feature} — {title}', loc='left')
            ax.grid(axis='y', alpha=.15)
    for ax in axes[-1]:
        ax.set_xlabel('Action-removal window (months relative to onset)')
    fig.supylabel('Reduction in persistence probability (percentage points)')
    export(fig, 'timed_ablation_intervals')
    plt.show()


## Alternative overview: heatmaps

The same means appear as colours, using one symmetric scale across phases. Each cell also prints its mean ± 95% interval half-width, so colour alone does not imply a resolved effect. Use this as an alternative to the interval figure rather than presenting the same results twice in the main paper.


In [6]:
if ready:
    limit = max(float(np.abs(mean).max()), 1.)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), sharey=True, layout='constrained')
    for col, (ax, title) in enumerate(zip(axes, ['El Niño', 'La Niña'])):
        im = ax.imshow(mean[col], cmap='RdBu_r', vmin=-limit, vmax=limit, aspect='auto')
        ax.set_xticks(np.arange(len(lags)), windows)
        ax.set_yticks(np.arange(len(features)), features)
        ax.set_title(f'{chr(97+col)}  {title}', loc='left')
        ax.set_xlabel('Action-removal window (months relative to onset)')
        for row in range(len(features)):
            for k in range(len(lags)):
                ax.text(k, row, f'{mean[col,row,k]:+.1f}\n±{ci[col,row,k]:.1f}',
                        ha='center', va='center', fontsize=9,
                        color='white' if abs(mean[col,row,k]) > .65*limit else 'black')
    fig.colorbar(im, ax=axes, shrink=.85, label='Reduction in persistence probability (percentage points)')
    export(fig, 'timed_ablation_heatmap')
    plt.show()
